# RL Beat Generation — Phase 2 Discriminator Training

Trains `BeatDiscriminator` (`num_instruments=8, d_ff=128`) on full 8-instrument Groove MIDI
grids (drums + melodic layers) vs. synthetic negatives.  Produces `discriminator_phase2.pt`,
which is saved to Drive and loaded by the PPO training notebook.

> **Runtime:** `Runtime → Change runtime type → T4 GPU`  
> **Run order:** Execute cells top-to-bottom. Drive mounts in Cell 2.

## 1 · Setup

In [ ]:
!git clone -b feature/phase2-updates https://github.com/Atharv-Girish-Chaudhary/rl-beat-generation.git
%cd rl-beat-generation
!pip install -e . -q

## 2 · Mount Drive + Copy Data

`groove_grids.npy` is too large to commit to git. This cell pulls it from Drive
so the training script can find it at `data/processed/groove_grids.npy`.

Expected shape after Phase 2 rebuild: `(N, 8, 16)` — all 8 instrument rows.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
DRIVE_PATH = "/content/drive/MyDrive/rl_term_project"

os.makedirs("data/processed", exist_ok=True)
if not os.path.exists("data/processed/groove_grids.npy"):
    shutil.copy(f"{DRIVE_PATH}/data/processed/groove_grids.npy",
                "data/processed/groove_grids.npy")
    print("✅ Copied groove_grids.npy from Drive")
else:
    print("✅ groove_grids.npy already present")

import numpy as np
shape = np.load("data/processed/groove_grids.npy").shape
print(f"   Shape: {shape}  (expected (N, 8, 16))")
assert shape[1] == 8 and shape[2] == 16, (
    f"Unexpected shape {shape} — run process_groove.py --phase 2 locally first."
)

## 3 · Device Check

In [ ]:
import torch

if torch.cuda.is_available():
    device = "cuda"
    props = torch.cuda.get_device_properties(0)
    print(f"GPU  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {props.total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    device = "mps"
    print("GPU  : Apple MPS")
else:
    device = "cpu"
    print("GPU  : none — training on CPU (will be slow)")

with open('/proc/meminfo') as f:
    lines = {k.strip(): v.strip()
             for k, v in (l.split(':', 1) for l in f if ':' in l)}
total_kb = int(lines.get('MemTotal', '0 kB').split()[0])
avail_kb = int(lines.get('MemAvailable', '0 kB').split()[0])
print(f"RAM  : {total_kb/1e6:.1f} GB total, {avail_kb/1e6:.1f} GB available")
print(f"\nPyTorch: {torch.__version__}")

## 4 · Config

Phase 2 target: validation accuracy **≥ 0.75** within 20 epochs on T4.
`PHASE = 2` routes the training script to use all 8 instrument rows and
saves to `discriminator_phase2.pt` — the Phase 1 checkpoint is never touched.

In [ ]:
PHASE      = 2
EPOCHS     = 20
BATCH_SIZE = 64
LR         = 1e-3
SAVE_PATH  = "outputs/checkpoints/discriminator_phase2.pt"

# L is derived from PHASE — never hardcoded
L = 4 if PHASE == 1 else 8
T = 16

print(f"Phase    : {PHASE}  (L={L}, T={T})")
print(f"Epochs   : {EPOCHS}")
print(f"Batch    : {BATCH_SIZE}")
print(f"LR       : {LR}")
print(f"Save to  : {SAVE_PATH}")

## 5 · Train

Calls `train_discriminator()` inline so config values above are forwarded.
Equivalent to `python scripts/train_discriminator.py --phase 2` locally.

With `--phase 2`, the script bypasses the `[:4]` slice and routes all 8 spatial
rows into the token embeddings. Best checkpoint is saved to `SAVE_PATH`.

In [ ]:
import sys, matplotlib
sys.path.insert(0, '.')
matplotlib.use('Agg')   # headless during training; plots saved to outputs/plots/

from scripts.train_discriminator import train_discriminator

train_discriminator(
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    phase=PHASE,
    save_path=SAVE_PATH,
)

## 6 · Sanity Check

Reloads the saved checkpoint and verifies a structured full-band beat
scores higher than silence. All tensor dimensions are derived from `L` / `T`
config — no hardcoded values.

In [ ]:
from beat_rl.models import BeatDiscriminator

disc = BeatDiscriminator(
    num_instruments=L, num_steps=T,
    d_model=64, num_heads=4, num_blocks=2, d_ff=128
).to(device)
disc.load_state_dict(torch.load(SAVE_PATH, map_location=device, weights_only=True))
disc.eval()
print(f"Loaded {SAVE_PATH}")

# ── Pass 1: completely silent grid ─────────────────────────────────────────
silent = torch.zeros(1, L, T, device=device)
with torch.no_grad():
    logit_s, _ = disc(silent)
score_silent = torch.sigmoid(logit_s).item()

# ── Pass 2: structured Phase 2 beat (drums + bass lock) ────────────────────
structured = torch.zeros(1, L, T)
structured[0, 0, [0, 4, 8, 12]] = 1.0          # kick   — quarter notes
structured[0, 1, [4, 12]]       = 1.0          # snare  — 2 & 4
structured[0, 2, list(range(0, 16, 2))] = 1.0  # hi-hat — 8th notes
structured[0, 3, [4, 12]]       = 1.0          # clap   — 2 & 4
if L == 8:
    structured[0, 4, [0, 4, 8, 12]] = 1.0      # bass   — locked to kick
with torch.no_grad():
    logit_r, _ = disc(structured.to(device))
score_structured = torch.sigmoid(logit_r).item()

print(f"\nSilent grid score    : {score_silent:.4f}  (expect < 0.5)")
print(f"Structured beat score: {score_structured:.4f}  (expect > silent)")

assert score_structured > score_silent, (
    f"Sanity check FAILED — structured ({score_structured:.4f}) "
    f"<= silent ({score_silent:.4f}). "
    "Re-train or check that val accuracy reached >= 0.75."
)
print("\n✅ Sanity check passed — discriminator ready for PPO.")

## 7 · Save to Drive

Copies the checkpoint to Drive so it persists across Colab sessions
and is available to the PPO training notebook.

In [ ]:
import shutil, os

dest_dir = f"{DRIVE_PATH}/checkpoints"
os.makedirs(dest_dir, exist_ok=True)

shutil.copy(
    SAVE_PATH,
    f"{dest_dir}/discriminator_phase2.pt",
)
print(f"✅ Saved to {dest_dir}/discriminator_phase2.pt")